# Spring of Code - Artificial Intelligence

## Week 09: Deep Learning Advanced

### Day 03: Video Action Recognition (RNN/GRU)

In this notebook, we will tackle **Video Action Recognition**. Videos are fundamentally sequences of images (frames). We will extract visual features from individual frames using a Convolutional Neural Network (CNN) and pass those sequences of features into a Recurrent Neural Network (GRU) to predict the action.


## 1. Environment Setup & Dependencies
We need `patool` to extract the `.rar` dataset programmatically and `opencv-python` to process video frames.


In [ ]:
!pip install patool opencv-python

## 2. Import Libraries


In [ ]:
import os
import cv2
import numpy as np
import urllib.request
import patoolib
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt

print(f"TensorFlow Version: {tf.__version__}")

## 3. Download and Extract the UCF101 Dataset
UCF101 is a large dataset. We will download the RAR file directly and extract it. This process might take a while if it is your first time running this cell.


In [ ]:
dataset_url = "https://www.crcv.ucf.edu/data/UCF101/UCF101.rar"
rar_filename = "UCF101.rar"
dataset_dir = "UCF-101"

# 1. Download
if not os.path.exists(rar_filename) and not os.path.exists(dataset_dir):
    print("Downloading UCF101 dataset (This is a 6.5GB file, please be patient)...")
    urllib.request.urlretrieve(dataset_url, rar_filename)
    print("Download complete!")
else:
    print("Dataset RAR file already exists or is already extracted.")

# 2. Extract
if os.path.exists(rar_filename) and not os.path.exists(dataset_dir):
    print("Extracting dataset...")
    patoolib.extract_archive(rar_filename, outdir=".")
    print("Extraction complete!")
else:
    print("Dataset folder is ready!")

## 4. Select a Subset of Classes
To ensure this notebook trains rapidly during a standard classroom session, we will focus on a subset of 4 visually distinct action classes.


In [ ]:
SELECTED_CLASSES = ['ApplyEyeMakeup', 'Archery', 'Biking', 'HorseRiding']
NUM_FRAMES = 20  # We will uniformly sample exactly 20 frames from every video, no matter how long it is

print(f"Target classes for this model: {SELECTED_CLASSES}")

## 5. Prepare the CNN Feature Extractor
Instead of training on raw pixels (which is extremely slow for video), we will pass each frame through a pre-trained **MobileNetV2** and extract a 1280-dimensional feature vector.


In [ ]:
# Load MobileNetV2 (headless)
cnn_extractor = tf.keras.applications.MobileNetV2(
    weights='imagenet',
    include_top=False,
    pooling='avg',
    input_shape=(224, 224, 3)
)

# We do NOT want to train the CNN
cnn_extractor.trainable = False

# Helper function to preprocess frames for MobileNetV2
def preprocess_frame(frame):
    frame = cv2.resize(frame, (224, 224))
    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    frame = tf.keras.applications.mobilenet_v2.preprocess_input(frame)
    return frame

## 6. Extract and Save Sequence Features Frame-wise
For each video, we will evenly space out `NUM_FRAMES` across the video's total duration. If the video is 10 seconds (300 frames), we still only grab 20 frames spread throughout to capture the entire action! 

The extracted sequences will be saved as `.npy` files with the same name as the video (e.g. `HorseRiding/v_HorseRiding_g01_c01.npy`).


In [ ]:
features_dir = "video_features"
os.makedirs(features_dir, exist_ok=True)

def extract_and_save_features(video_path, save_path):
    # Open the video
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    # Safety check: if video is unreadable or too short
    if total_frames < NUM_FRAMES:
        cap.release()
        return False
        
    # Evenly space the frames across the whole video duration
    frame_indices = np.linspace(0, total_frames - 1, NUM_FRAMES, dtype=int)
    
    frames = []
    for idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(preprocess_frame(frame))
        
    cap.release()
    
    # If we successfully captured 20 frames, process them
    if len(frames) == NUM_FRAMES:
        frames_array = np.array(frames) # Shape: (20, 224, 224, 3)
        # Extract features through MobileNetV2 in one batch
        features = cnn_extractor.predict(frames_array, verbose=0) # Shape: (20, 1280)
        
        # Save the frame-wise features (.npy)
        np.save(save_path, features)
        return True
    return False

# Execute Extraction for the Selected Classes
print("Starting feature extraction... This might take a few minutes.")
for class_name in SELECTED_CLASSES:
    class_dir = os.path.join(dataset_dir, class_name)
    save_class_dir = os.path.join(features_dir, class_name)
    os.makedirs(save_class_dir, exist_ok=True)
    
    if os.path.exists(class_dir):
        videos = os.listdir(class_dir)
        count = 0
        for video_name in videos:
            if video_name.endswith('.avi'):
                video_path = os.path.join(class_dir, video_name)
                save_path = os.path.join(save_class_dir, video_name.replace('.avi', '.npy'))
                
                # Only extract if we haven't already saved it
                if not os.path.exists(save_path):
                    success = extract_and_save_features(video_path, save_path)
                    if success:
                        count += 1
        print(f"Processed {count} new videos for class: {class_name}")
    else:
        print(f"Warning: Class directory {class_dir} not found. Did you extract the dataset?")
print("Feature extraction complete!")

## 7. Build X and Y for Training
We now load the saved `.npy` files. Each file contains a sequence of 20 frames, with 1280 features per frame.


In [ ]:
X = []
y = []

for label_idx, class_name in enumerate(SELECTED_CLASSES):
    save_class_dir = os.path.join(features_dir, class_name)
    if os.path.exists(save_class_dir):
        npy_files = os.listdir(save_class_dir)
        for npy_file in npy_files:
            feature_sequence = np.load(os.path.join(save_class_dir, npy_file))
            X.append(feature_sequence)
            y.append(label_idx)

X = np.array(X)  # Expected shape: (Num_Videos, 20, 1280)
y = np.array(y)

print(f"Feature matrix X shape: {X.shape}")
print(f"Labels y shape: {y.shape}")

## 8. Train/Test Split


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training sequences: {X_train.shape[0]}, Testing sequences: {X_test.shape[0]}")

## 9. Build the Sequence Model (GRU)
Now we use `tf.keras.Sequential` to build our Recurrent Neural Network. The GRU layer will read the frame-wise features sequentially and remember patterns over time to predict the action.


In [ ]:
model = models.Sequential([
    # Input is sequence of 20 frames, 1280 features each
    layers.Input(shape=(NUM_FRAMES, 1280)),
    
    # The GRU layer models the sequence
    layers.GRU(128, return_sequences=False),
    
    # Add Dropout for regularization
    layers.Dropout(0.5),
    
    # Final Classification Head
    layers.Dense(64, activation='relu'),
    layers.Dense(len(SELECTED_CLASSES), activation='softmax')
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

## 10. Train the Network


In [ ]:
print("Training the GRU model...")
history = model.fit(
    X_train, y_train,
    epochs=15,
    batch_size=16,
    validation_data=(X_test, y_test)
)
print("Training completed!")

## 11. Predict Action from ANY Video
This is a complete pipeline function. You can pass the path to ANY video (e.g. `.mp4`, `.avi`, `.mov`) of ANY length. It will uniformly sample 20 frames across the video duration, run them through the CNN, and pass them to our GRU to give a prediction!


In [ ]:
def predict_video_action(video_path):
    if not os.path.exists(video_path):
        print(f"Error: Video {video_path} not found.")
        return
    
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    if total_frames < NUM_FRAMES:
        print(f"Error: Video is too short. Must have at least {NUM_FRAMES} frames.")
        cap.release()
        return
        
    # Sample exactly NUM_FRAMES across the entire video
    frame_indices = np.linspace(0, total_frames - 1, NUM_FRAMES, dtype=int)
    frames = []
    
    for idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(preprocess_frame(frame))
    cap.release()
    
    if len(frames) == NUM_FRAMES:
        frames_array = np.array(frames)
        # 1. CNN Feature Extraction
        features = cnn_extractor.predict(frames_array, verbose=0)
        features_batch = np.expand_dims(features, axis=0) # Add batch dim: (1, 20, 1280)
        
        # 2. GRU Prediction
        pred_probs = model.predict(features_batch, verbose=0)
        pred_idx = np.argmax(pred_probs[0])
        pred_class = SELECTED_CLASSES[pred_idx]
        confidence = pred_probs[0][pred_idx] * 100
        
        print("-" * 40)
        print(f"Video Analysis Complete!")
        print(f"Predicted Action : **{pred_class}**")
        print(f"Confidence       : {confidence:.2f}%")
        print("-" * 40)
    else:
        print("Failed to extract enough frames from the video.")

## Test the Predictor


In [ ]:
# Test it on a video from the dataset
# (Make sure to point to a valid video path after extracting!)
test_video = os.path.join(dataset_dir, SELECTED_CLASSES[0], os.listdir(os.path.join(dataset_dir, SELECTED_CLASSES[0]))[0])
print(f"Testing on video: {test_video}")
predict_video_action(test_video)